<h2>
  Vignette 06: Spatial Analysis Tools<br>
  <span style="font-size: 12px; font-weight: normal; display: block">
    <i>"I wish the world was flat like the old days, Then I could travel just by folding a map" — Death Cab for Cutie</i><br>
  </span>
</h2>

In order to allow for examining spatial datasets, platpy includes a set of simple spatial tools to
segment an ROI into layers.  This includes three distinct methods of segmentation:

1. 'build_axis_bins()' segments an ROI into equally-spaced layers by slicing at a defined angle.  This tool is particularly useful for
examining gradients of expression - i.e. is a transcript distributed in a dorsal-ventral or medial-lateral gradient?

2. 'build_contour_shells()' segments an ROI into "onion skin" layers, to examine "inside out" gradients.

3. 'build_depth_bins()' is a hybrid of both of the above - this segments an ROI into equally spaced layers, along the contours of the parent.  This is particularly useful for dividing cortex samples, but may be useful for other anatomical features as well

In the next cells, we will load in the sample Xenium dataset, and also load in sample ROIs.  The ROIs were generated using the Napari approach described previously and saved as JSON files.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.path import Path as MplPath
import scanpy as sc

from platpy.io_utils import load_sample
from platpy.xenium_loader import load_xenium
from platpy.plot_utils import plot_image_grid, PlotObject
from platpy.channels import ROIChannel
from platpy.viewport import ViewPort
from platpy.cell_utils import build_cells_xenium
from platpy.spatial_tools import (
    extract_edges_from_endcaps, build_depth_bins, build_axis_bins,
    build_contour_shells, assign_contour_shells, assign_bins,
    bins_to_roi_channel, load_rois, get_cells_in_roi,
    annotate_roi_membership, bin_expression_matrix, polygon_area_um2
)

# --- UPDATE THESE PATHS ---
save_dir = Path.home() / "platpy_data" / "XeniumSamples"

sample = load_sample(str(save_dir / "WT_02"))
sample.view_defaults = {'rotation_deg': 105, 'flip_y': True}

print(f"Images: {list(sample.images.keys())}")
print(f"Omics:  {list(sample.omics.keys())}")
tx_obs = sample.omics['transcripts'].data.obs
print(f"Transcript table: {tx_obs.shape[0]:,} rows  |  columns: {list(tx_obs.columns)}")

# Build a cells x genes AnnData from the Xenium transcript assignments.
adata = build_cells_xenium(sample)
print(adata)

In [ ]:
########
## Load all analysis ROIs - These were drawn in Napari and saved as JSON files as shown in previous vignette.
## Then we'll extract all the cells in each ROI for analysis.
########
rois        = load_rois("demo/WT_02_binning_rois.json")
region_rois = load_rois("demo/WT_02_region_rois.json")

ctx_poly      = np.array(rois["ctx_poly"])
striatum_poly = np.array(region_rois["striatum"])

# Extract the subset of cells for each analysis ROI for downstream analysis
adata_ctx = get_cells_in_roi(adata, ctx_poly)
adata_str = get_cells_in_roi(adata, striatum_poly)
print(f"Cortex cells:   {adata_ctx.n_obs:,}")
print(f"Striatum cells: {adata_str.n_obs:,}")

# Overview: Show the ROIs on the DAPI image, correctly oriented.
vp = ViewPort.from_image(sample.images['DAPI'])
vp.update_view(center_um=(8800, 2500), fov_um=(3500, 3000), rotation_deg= 105, flip_y= True)
vp.add_layer(sample.images['DAPI']).adjust_layer(tint='xkcd:white', percentile_clip=(1, 99))
vp.add_layer(ROIChannel(name='cortex',   polygons={'cortex':   ctx_poly.tolist()  }).set_style(line_color='xkcd:cyan',          line_width=10))
vp.add_layer(ROIChannel(name='striatum', polygons={'striatum': striatum_poly.tolist()}).set_style(line_color='xkcd:bright orange', line_width=10))

plot_image_grid(vp.render(), titles=['Analysis ROIs — cortex (cyan)  ·  striatum (orange)'], width=4);

#### Axis bins (simple slices across an ROI)

To show whether a transcript is distributed across a simple gradient (e.g. dorsal-ventral), 
we can just slice an ROI into equal width bins.  For this, we can use the function 
`build_axis_bins`, which does just that.  We just need to supply this function the axis direction for
making the slices.  

Here we use the **striatum** ROI from Vig 5. We'll slice it into bins to assess D-V gradients. 
Calb1 (calbindin) is known to be expressed at higher levels in ventral striatum; we'll compare
that with Txnrd1, a metabolic gene with no known D-V bias.  

In [ ]:
## Generate bins along the D-V axis of the striatum ROI using build_axis_bins.
## First, let's make a viewport to look at our image

vp_bins = ViewPort.from_image(sample.images['DAPI'])
vp_bins.update_view(center_um=(9250, 1900), fov_um=(2500, 2000), rotation_deg= 105, flip_y= True)
vp_bins.add_layer(sample.images['DAPI']).adjust_layer(tint='xkcd:white', percentile_clip=(1, 99))

angle = vp_bins.screen_direction_to_angle(0)

n_axis_bins = 6
striatum_bins = build_axis_bins(striatum_poly, n_bins=n_axis_bins, angle_deg=angle)
print(f"Generated {len(striatum_bins)} striatum axis bins")

striatum_bin_chan = bins_to_roi_channel(striatum_bins, name='striatum_bins', 
                                        cmap_name='RdYlBu', label_color='white', label_size=1, label_weight=2)
vp_bins.add_layer(striatum_bin_chan)
plot_image_grid(vp_bins.render(), titles=['Striatum axis bins (0 = Ventral)'], width=5)

str_coords = adata_str.obsm['spatial']
dv_genes = [g for g in ['Calb1','Txnrd1'] if g in adata.var_names]  
print(f"D-V markers in panel: {dv_genes}")

adata_str_binned = assign_bins(adata_str.copy(), striatum_bins, col='axis_bin')

if dv_genes:
    mean_expr = np.zeros((n_axis_bins, len(dv_genes)))
    for b in range(n_axis_bins):
        cells_b = adata_str_binned[adata_str_binned.obs['axis_bin'] == b]
        if cells_b.n_obs > 0:
            mean_expr[b] = np.asarray(cells_b[:, dv_genes].X.mean(axis=0)).flatten()
    x = np.arange(n_axis_bins)
    width = 0.35
    fig, ax = plt.subplots(figsize=(7, 4))
    for j, (gene, col) in enumerate(zip(dv_genes, ['steelblue', 'coral'])):
        ax.bar(x + j*width - width/2, mean_expr[:, j], width, label=gene, color=col, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f'Bin {i}' for i in range(n_axis_bins)])
    ax.set_xlabel('Axis bin (ventral → dorsal)')
    ax.set_ylabel('Mean transcripts / cell')
    ax.set_title('Striatum: Calb1 (ventral) vs Txnrd1 (non-graded) across d-v axis')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
#### Part 2 — Contour shells ('onion skin' binning)

`build_contour_shells` generates an evenly-spaced set of bins that retain the overall
shape of the structure.  This is useful for structures where expression is expected to 
be higher or lower in a central core vs. the outer edges.  This may be of particular
use in structures such as tumors or ganglia.

We will test this function on the striatum as well.

**Note:** the bottom edge of this ROI is not actually the edge of the striatum - it is
the edge of the tissue slice. In this example, we will create a new ROI that approximates
what the complete striatum would look like, genearate the 'onion skin' shells, then we
will crop our ROIs to the tissue slice.  This allows the binning to reflect the true 
anatomical structure.

In [ ]:
crop_rois        = load_rois("demo/WT_02_striatum_crop.json")
striatum_full_np = np.array(crop_rois["striatum_full"])
tissue_edge_roi  = ROIChannel(name="tissue_edge", polygons={"edge": np.array(crop_rois["tissue_edge_crop"])})

n_shells   = 5
shells_str = build_contour_shells(striatum_full_np, n_shells=n_shells)
print(f"Generated {len(shells_str)} shells")

vp_sh = ViewPort.from_image(sample.images["DAPI"])
vp_sh.update_view(center_um=(9250, 1900), fov_um=(2500, 2000), rotation_deg=105, flip_y=True)
vp_sh.update_view(**sample.view_defaults)

shells_chan = bins_to_roi_channel(
    [outer for outer, _ in shells_str], name="striatum_shells",
    cmap_name="plasma", line_color="white", line_width=5,
    label_color="white", label_size=0.8, label_weight=2,
    label_anchor=vp_sh.screen_direction_to_angle("top")
).crop(tissue_edge_roi)

vp_sh.add_layer(sample.images["DAPI"]).adjust_layer(tint="xkcd:white", percentile_clip=(1, 99))
vp_sh.add_layer(shells_chan)
plot_image_grid(vp_sh.render(), titles=[f"Striatum: {len(shells_str)} contour shells (0 = outermost, cropped)"], width=5);

In [ ]:
# Calculate the per-cell mean expression of two genes of interest across the shells.
target_genes = [g for g in ["Nnat", "Vat1l"] if g in adata_str.var_names]
print(f"Target genes in panel: {target_genes}")

# Notes:
#  - We are creating an adata that has the striatum cells assigned to shells.
#  - We wil use the cropped shell polygons for assignment.
#  - The shells are nested, so we start with the larger (outer) shells and work inward.
#  - We use overwrite mode as we assign shells, and becasuse the assignment goes
#    outer→inner, each cell ends up labeled with its innermost (most specific) shell.

adata_str_sh = annotate_roi_membership(
    adata_str.copy(), shells_chan.polygons, col="shell", cell_duplicates="overwrite"
)
shell_keys = list(shells_chan.polygons.keys())  # ["bin_0", "bin_1", ...]
n_sh = len(shell_keys)
print(f"Cells assigned: {(adata_str_sh.obs['shell'] != 'unassigned').sum():,} / {adata_str_sh.n_obs:,}")

#########
## This is a standard scanpy workflow to extract the mean expression from each shell and plot it.
#########

mean_expr = np.zeros((n_sh, len(target_genes)))
for i, skey in enumerate(shell_keys):
    cells_s = adata_str_sh[adata_str_sh.obs["shell"] == skey]
    if cells_s.n_obs > 0:
        mean_expr[i] = np.asarray(cells_s[:, target_genes].X.mean(axis=0)).flatten()

x     = np.arange(n_sh)
width = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
for j, (gene, col) in enumerate(zip(target_genes, ["steelblue", "coral"])):
    ax.bar(x + j*width - width/2, mean_expr[:, j], width, label=gene, color=col, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f"Shell {i}" for i in range(n_sh)])
ax.set_xlabel("Shell (0 = outermost)")
ax.set_ylabel("Mean transcripts / cell")
ax.set_title("Striatum: Nnat vs Vat1l across contour shells")
ax.legend()
plt.tight_layout()
plt.show()


#### Part 3 — Depth bins (curvature-aware cortical layers)

The depth bin function is designed to generate evenly spaced layers from an roi, while
retaining the curvature of the structure.  This feature was designed for annotating 
cerebral cortex, but it may be useful for other layered structures both inside and
outside of the brain (including structures such as skin, or tumor invasion assays).

To generate the layers, we will start with the ROI for the entire cortex, and then we will
define a "slice" that gets layered.  Each slice is defined by two "endcap" lines, oriented
from top (pia) to bottom (white matter).  We have generated these lines in our Napari
function; we have defined two separate "slices", medial and lateral, using two pairs of end
cap lines (`medial_cap`/`medial_cap2`, `lateral_cap`/`lateral_cap2`).  The 
`extract_edges_from_endcaps` function extracts out the top and bottom segments of the 
cortex ROI, then the `build_depth_bins` interpolates a series of slice boundaries that 
follow the contours of the cortex.  The rest of the workflow is parallel to the
simpler functions described above.

In [ ]:
# These are the lines we will use for our endcaps.
medial_cap1 = np.array(rois["medial_cap"])
medial_cap2 = np.array(rois["medial_cap2"])
lateral_cap1 = np.array(rois["lateral_cap"])
lateral_cap2 = np.array(rois["lateral_cap2"])

# Now we will used these endcap lines to define the top and bottom edges of the ROI 
# that will define our cortical bins

pial_med, wm_med = extract_edges_from_endcaps(ctx_poly, medial_cap1, medial_cap2)
pial_lat, wm_lat = extract_edges_from_endcaps(ctx_poly, lateral_cap1, lateral_cap2)

# Verify: pial (blue) should be the outer/superficial edge, WM (red) the inner/deep edge.
#   (We'll plot this out using standard matplotlib; this isn't in a viewport so it will be in micron-space))

coords = adata_ctx.obsm["spatial"]  # cortex cell positions to plot as a background in our image

fig, axes = plt.subplots(1, 2, figsize=(9, 7), sharey=True)
for ax, pial, wm, title in zip(
        axes,
        [pial_med, pial_lat],
        [wm_med,   wm_lat],
        ['Medial strip', 'Lateral strip']):
    ax.scatter(coords[:, 0], coords[:, 1], s=0.3, c='lightgray', alpha=0.2, rasterized=True)
    ax.plot(pial[:, 0], pial[:, 1], 'b-', linewidth=2, label='Pial')
    ax.plot(wm[:, 0],   wm[:, 1],   'r-', linewidth=2, label='WM')
    ax.scatter(pial[0, 0], pial[0, 1], s=60, c='blue', zorder=5)
    ax.scatter(wm[0, 0],   wm[0, 1],   s=60, c='red',  zorder=5)
    ax.legend()
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_xlabel('x (µm)')
    ax.set_title(title)

axes[0].set_ylabel('y (µm)')
plt.suptitle('Edge extraction — verify pial (blue) vs WM (red)')
plt.tight_layout()
plt.show()

In [ ]:
n_bins = 10
depth_bins_med = build_depth_bins(pial_med, wm_med, n_bins=n_bins, n_interp=100)
depth_bins_lat = build_depth_bins(pial_lat, wm_lat, n_bins=n_bins, n_interp=100)
print(f"Medial: {len(depth_bins_med)} bins  |  Lateral: {len(depth_bins_lat)} bins")

med_ctx_bin_rois = bins_to_roi_channel(depth_bins_med, name="depth_bins_med",
                               cmap_name="RdYlBu", line_color="white", line_width=1,
                               label_color="white", label_size=0.8, label_weight=2,
                               label_anchor=None)

lat_ctx_bin_rois = bins_to_roi_channel(depth_bins_lat, name="depth_bins_lat",
                               cmap_name="RdYlBu", line_color="white", line_width=1,
                               label_color="white", label_size=0.8, label_weight=2,
                               label_anchor=None)

vp_ctx = ViewPort.from_image(sample.images["DAPI"])
vp_ctx.update_view(center_um=(8800, 2500), fov_um=(3500, 3000), rotation_deg=105, flip_y=True)

vp_med = vp_ctx.clone(keep_layers=False)
vp_med.add_layer(sample.images["DAPI"]).adjust_layer(tint="xkcd:white", percentile_clip=(1, 99))
vp_med.add_layer(med_ctx_bin_rois)

vp_lat = vp_ctx.clone(keep_layers=False)
vp_lat.add_layer(sample.images["DAPI"]).adjust_layer(tint="xkcd:white", percentile_clip=(1, 99))
vp_lat.add_layer(lat_ctx_bin_rois)

plot_image_grid([vp_med.render(), vp_lat.render()],
               titles=["Medial depth bins (0 = pial)", "Lateral depth bins (0 = pial)"],
               width=5, show_axis=False, font_size='small');

In [ ]:
## Now to begin the analysis: 
## Extract the cells within each bin into a separate adata, with a 'depth_bin' column that marks bin assignment.

adata_med = annotate_roi_membership(
    adata_ctx.copy(), depth_bins_med, col="depth_bin", cell_duplicates="overwrite"
)
adata_med = adata_med[adata_med.obs["depth_bin"] != "unassigned"].copy()  # filter out unassigned cells

adata_lat = annotate_roi_membership(
    adata_ctx.copy(), depth_bins_lat, col="depth_bin", cell_duplicates="overwrite"
)
adata_lat = adata_lat[adata_lat.obs["depth_bin"] != "unassigned"].copy()  # filter out unassigned cells

print(f"Medial strip:  {adata_med.n_obs:,} cells")
print(adata_med.obs["depth_bin"].value_counts().sort_index())
print(f"Lateral strip: {adata_lat.n_obs:,} cells")
print(adata_lat.obs["depth_bin"].value_counts().sort_index())


In [ ]:
#########
##  Assay the expression of layer markers across the bins, to visualize expression of
##  genes known to be expressed in upper vs. lower layers of the cortex.
##  This is mostly a standard scanpy workflow, with an addition of a small helper function.
#########

## Assay a subset of cortical layer markers that are present in this Xenium panel.
## (These are roughly in order from upper to lower layer markers) 
layer_markers = ['Cux1', 'Cux2', 'Rorb', 'Bcl11b', 'Foxp2', 'Rbp4', 'Nrn1', 'Fezf2', 'Reln']

## Create a heat-map of mean expression per bin for the list of markers.
## This is standard scanpy, but we've included a small helper function bin_expression_matrix()
## for this common workflow procedure.
e_med, bin_keys = bin_expression_matrix(adata_med, layer_markers, bin_col="depth_bin")
e_lat, _        = bin_expression_matrix(adata_lat, layer_markers, bin_col="depth_bin")
n_bins = len(bin_keys)

# Normalize each gene to its global max across both strips so scales are comparable.
global_max  = np.maximum(e_med.max(axis=0), e_lat.max(axis=0)) + 1e-6
e_med_norm  = e_med / global_max
e_lat_norm  = e_lat / global_max

## Plot the results using standard matplotlib functions.
bin_labels       = [f"Bin {i}" for i in range(n_bins)]
bin_labels[0]   += "(pial)"
bin_labels[-1]  += "(WM)"

fig, axes = plt.subplots(2, 1, figsize=(max(4, len(layer_markers) * 1), 5),
                         sharey=True, sharex=True)
for ax, expr_norm, title in zip(axes, [e_med_norm, e_lat_norm], ["Medial", "Lateral"]):
    im = ax.imshow(expr_norm.T, aspect="auto", cmap="RdYlBu_r", vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, label="Relative expression")
    ax.set_xticks(range(n_bins))
    #ax.set_xticklabels(bin_labels)
    ax.set_yticks(range(len(layer_markers)))
    ax.set_yticklabels(layer_markers)
    ax.set_xlabel("Depth bin (pial → WM)")
    ax.set_title(title)

fig.suptitle("WT_02: cortical layer markers by depth — medial vs lateral")
plt.tight_layout()
plt.show()

#### Part 4 — Additional ways to visualize binned data: the 'Unfolded Cortex' map

To better visualize layering, we're adding a function that creates an 'unfolded'
version of a curved section of cortex. The result a rectangular map that is ideal
for visualizing layering, and comparing between different regions.

Note: Just like with a 2-D map of the earth, there are distortions generated by 
projecting curved geometry onto a rectangular space; the layering relationship is
changed, but the horizontal geometry is stretched or squeezed. 

In [ ]:
## Use the 'unfold_cortex' function to generate a map of cortical cells assigned to each layer.
from platpy.spatial_tools import unfold_cortex

adata_med_u = unfold_cortex(adata_med, pial_med, wm_med, n_lat=200)
adata_lat_u = unfold_cortex(adata_lat, pial_lat, wm_lat, n_lat=200)

print(f"Medial:  {adata_med_u.n_obs:,} cells unfolded")
print(f"Lateral: {adata_lat_u.n_obs:,} cells unfolded")

# Plot Cux1 + Cux2 expression in the unfolded cortex.
# Sum both if present; fall back to whichever is in the panel.
plot_genes = [g for g in ['Cux1', 'Cux2'] if g in adata_ctx.var_names]
print(f"Plotting: {plot_genes}")

def get_expr_sum(adata_u, genes):
    return np.asarray(adata_u[:, genes].X.sum(axis=1)).flatten()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, adata_u, title in zip(axes,
                               [adata_med_u, adata_lat_u],
                               ['Medial', 'Lateral']):
    expr = get_expr_sum(adata_u, plot_genes)
    vmax = np.percentile(expr, 99)
    sc = ax.scatter(
        adata_u.obs['lat_coord'],
        adata_u.obs['depth_coord'],
        c=expr, cmap='cividis_r', s=6, alpha=0.8,
        vmin=0, vmax=vmax, rasterized=True
    )
    ax.invert_yaxis()          # pial at top, WM at bottom
    ax.set_ylabel('Depth\n(pial → WM)')
    ax.set_title(f'{title}  —  {" + ".join(plot_genes)}')
    ax.set_aspect('auto')
    plt.colorbar(sc, ax=ax, label='Transcripts')

axes[-1].set_xlabel('Lateral position (cap1 end → cap2 end)')
plt.suptitle('Unfolded cortex — upper layer markers')
plt.tight_layout()
plt.show()

#### Part 5 — Density histogram

The unfolded cortex map illustrates that there are differences in the numbers of
upper vs. lower layer cortex in the medial vs. lateral cortex.  Let's explore this
further using a simple histogram plot.  This is mostly straighforward scanpy code;
with a few functions added from platpy.  This plot shows that the tight cell packing
we see in the upper layers of the lateral cortex isn't just artifact.

In [ ]:
# Let's calculate cell density of upper-layer cells across the cortical bins.
# We'll use expression of cux1 and Cux2 as a proxy for upper layer identity.
# We'll count the number of Cux1/2+ cells per bin, and divide by the area
# of the bin using the polygon_area_um2() function from spatial_tools.py

cux_genes     = [g for g in ['Cux1', 'Cux2'] if g in adata_ctx.var_names]
cux_threshold = 5

results = {}
for label, adata_b, bins in [('Medial', adata_med, depth_bins_med),
                              ('Lateral', adata_lat, depth_bins_lat)]:
    density, cux_frac = [], []
    for b in range(n_bins):
        cells_b = adata_b[adata_b.obs['depth_bin'] == f'bin_{b}']
        area = polygon_area_um2(np.array(bins[b]))
        density.append(cells_b.n_obs / area if area > 0 else 0)
        if cux_genes and cells_b.n_obs > 0:
            cux_sum = np.asarray(cells_b[:, cux_genes].X.sum(axis=1)).flatten()
            cux_frac.append((cux_sum > cux_threshold).mean() * 100)
        else:
            cux_frac.append(0.0)
    results[label] = dict(density=density, cux_frac=cux_frac)


# We'll just plot this as a histogram using standard matplotlib
x = np.arange(n_bins)
width = 0.35
xtick_labels = [f'Bin {i}' for i in range(n_bins)]
xtick_labels[0] += '\n(pial)'; xtick_labels[-1] += '\n(WM)'

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for j, (label, col) in enumerate([('Medial', 'steelblue'), ('Lateral', 'coral')]):
    r = results[label]
    axes[0].bar(x + j*width - width/2, r['density'],  width, label=label, color=col, alpha=0.85)
    axes[1].bar(x + j*width - width/2, r['cux_frac'], width, label=label, color=col, alpha=0.85)

for ax, ylabel, title in zip(
        axes,
        ['Cells / µm²', f'% cells with {"+".join(cux_genes)} > {cux_threshold} tx'],
        ['Cell density per bin', f'{"+".join(cux_genes)}+ fraction']):
    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels, fontsize=8)
    ax.set_xlabel('Depth bin')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()

plt.suptitle('WT_02 — cortical depth bin quantification')
plt.tight_layout()
plt.show()

####Spatial geometry as an analytical tool
In this example, you may have noticed a few key things:

- **Three complementary binning strategies, one consistent API.** Axis bins slice a structure into simple equal-width strips for gradient analysis. Contour shells erode inward to capture edge-to-core gradients. Depth bins follow the actual curvature of a tissue surface — useful wherever layers matter but the boundary isn't flat. All three produce the same output: a set of polygons in micron space that can be used to assign cells and quantify expression.
- **Bins are a first-class visualization object.** The bins_to_roi_channel function converts any set of bin polygons into a color-coded ViewPort layer with labeled centroids. You see the geometry and the biology in the same frame, correctly oriented, with no separate plotting step.
- **ROIs are flexible.** Simple functions for working with multiple ROIs are built in: Union, difference, and intersections can be used to make new ROIs.  We used this to crop the contour shell ROI at a tissue boundary to reflect true anatomical structure. This can be used to quickly make a complex analysis: draw a pair of boxes to define medial and lateral, and you can apply those to all the other ROIs in your analysis.
- **Curved structures can be unfolded.** The cortex unfolding maps a curved tissue strip onto a flat 2D coordinate system where laminar structure runs top-to-bottom and lateral position runs left-to-right — making it easier to compare across samples or visualize gradients across the full extent of a region.

**Next:** In the final vignette, we'll show how fine sub-pixel alignment between imaging channels is handled — bringing everything into the sharpest possible spatial correspondence before downstream analysis.